# Ejemplo 1 · Encontrar e identificar imágenes
**Estud-IA · Clase 5 · Intermedio.** Tú decides, la IA escribe, tú ejecutas y mides.

**Problema.** Tienes un catálogo de imágenes y quieres dos cosas: (A) dado una imagen, **encontrar las más parecidas**, y (B) **identificar qué contiene cada imagen** para saber qué imágenes tienes.

**Datos.** Para que el notebook corra en cualquier computador usamos imágenes de dígitos escritos a mano que ya vienen con scikit-learn (1.797 imágenes de 8×8 píxeles, 10 clases). Es un catálogo de práctica, no un caso de negocio: el método es el mismo con tus fotos.

**Los modelos, en palabras simples**
- **Imagen = números.** Cada imagen de 8×8 son 64 números (0 es blanco, 16 es negro). Para el computador una imagen es una lista de números.
- **Buscar parecidas (no se entrena nada):** se mide qué tan parecidas son dos listas de números (similitud del coseno) y se devuelven las más cercanas.
- **Identificar (se entrena):** regresión logística. Aprende un peso por cada píxel y por cada clase; la clase con más puntos gana. Entrenar es ajustar esos pesos con ejemplos etiquetados.

## Parte 0 · Preparar y mirar las imágenes

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

SEMILLA = 1540
d = load_digits()
X, y, img = d.data, d.target, d.images          # X: 64 números por imagen
print("Imágenes:", X.shape[0], "| números por imagen:", X.shape[1], "| clases:", len(set(y)))

fig, ax = plt.subplots(2, 8, figsize=(12, 3.2))
for a, i in zip(ax.ravel(), range(16)):
    a.imshow(img[i], cmap="gray_r"); a.set_title(f"clase {y[i]}"); a.axis("off")
plt.show()
print("Así ve el computador la primera imagen:\n", img[0].astype(int))

## Parte 1 · Separar los datos (la regla de siempre)
Entrenamiento para aprender, validación para medir, prueba que **no se abre** hasta el final. La partición se hace una sola vez y no se cambia.

In [ ]:
idx = np.arange(len(X))
i_ent, i_resto = train_test_split(idx, test_size=0.4, random_state=SEMILLA, stratify=y)
i_val, i_prueba = train_test_split(i_resto, test_size=0.5, random_state=SEMILLA, stratify=y[i_resto])
print("Entrenamiento:", len(i_ent), "| Validación:", len(i_val), "| Prueba (sin abrir):", len(i_prueba))

## Parte 2 · A: encontrar imágenes parecidas (sin entrenar)
**Qué hace:** toma una imagen de consulta y devuelve las 5 más parecidas del catálogo (el catálogo es el conjunto de entrenamiento).
**Qué debes ver:** las imágenes devueltas se parecen a la consulta.

In [ ]:
from sklearn.neighbors import NearestNeighbors

catalogo = X[i_ent]
buscador = NearestNeighbors(n_neighbors=5, metric="cosine").fit(catalogo)

def parecidas(i_consulta):
    dist, pos = buscador.kneighbors(X[i_consulta].reshape(1, -1))
    return i_ent[pos[0]]

for q in i_val[:3]:
    ids = parecidas(q)
    fig, ax = plt.subplots(1, 6, figsize=(10, 2))
    ax[0].imshow(img[q], cmap="gray_r"); ax[0].set_title(f"consulta ({y[q]})"); ax[0].axis("off")
    for a, j in zip(ax[1:], ids):
        a.imshow(img[j], cmap="gray_r"); a.set_title(f"clase {y[j]}"); a.axis("off")
    plt.show()

**Cómo medirlo (k de N):** de las 5 imágenes devueltas, ¿cuántas son de la misma clase que la consulta? Se calcula para todas las consultas de validación.

In [ ]:
aciertos = 0; total = 0
for q in i_val:
    ids = parecidas(q)
    aciertos += int((y[ids] == y[q]).sum()); total += len(ids)
print(f"Búsqueda: {aciertos} de {total} imágenes devueltas son de la misma clase ({aciertos/total:.2f})")

## Parte 3 · B: identificar qué contiene cada imagen (se entrena)
**Qué hace:** entrena una regresión logística con las imágenes de entrenamiento y mide en las de validación.
**Qué debes ver:** un «k de N» en imágenes que el modelo no vio al aprender.

In [ ]:
from sklearn.linear_model import LogisticRegression

modelo = LogisticRegression(max_iter=2000)
modelo.fit(X[i_ent], y[i_ent])
pred_val = modelo.predict(X[i_val])
k = int((pred_val == y[i_val]).sum()); N = len(i_val)
print(f"Identificar: {k} de {N} aciertos en validación ({k/N:.3f})")

### ¿Dónde se equivoca?
Las confusiones dicen qué clases se parecen entre sí. Mira las imágenes que falló: ¿las confundirías tú?

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd
cm = pd.DataFrame(confusion_matrix(y[i_val], pred_val), index=[f"real {i}" for i in range(10)], columns=[f"pred {i}" for i in range(10)])
display(cm)
mal = np.where(pred_val != y[i_val])[0][:8]
fig, ax = plt.subplots(1, max(len(mal),1), figsize=(1.6*max(len(mal),1), 2))
ax = np.atleast_1d(ax)
for a, p in zip(ax, mal):
    j = i_val[p]; a.imshow(img[j], cmap="gray_r"); a.set_title(f"real {y[j]}\npred {pred_val[p]}", fontsize=8); a.axis("off")
plt.show()

## Parte 4 · Cómo se entrena: qué aprendió el modelo
El modelo aprende un peso por píxel y por clase. Si los dibujamos como imagen, se ve **a qué píxeles le presta atención** cada clase (claro = suma puntos, oscuro = resta).

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(10, 4.4))
for c_, a in zip(range(10), ax.ravel()):
    a.imshow(modelo.coef_[c_].reshape(8, 8), cmap="coolwarm"); a.set_title(f"clase {c_}"); a.axis("off")
plt.show()

### ¿Más imágenes de entrenamiento ayudan?
Se entrena con 10 %, 25 %, 50 % y 100 % de los datos de entrenamiento y se mide en la **misma** validación. Cambia una sola cosa: la cantidad de ejemplos.

In [ ]:
rng = np.random.default_rng(SEMILLA)
filas = []
for frac in [0.10, 0.25, 0.50, 1.0]:
    n = max(int(len(i_ent) * frac), 10)
    sub = rng.choice(i_ent, size=n, replace=False)
    m = LogisticRegression(max_iter=2000).fit(X[sub], y[sub])
    k = int((m.predict(X[i_val]) == y[i_val]).sum())
    filas.append({"imágenes de entrenamiento": n, "aciertos en validación": f"{k} de {len(i_val)}"})
display(pd.DataFrame(filas))

## Parte 5 · «Identificar cuáles imágenes tengo»: inventario
Con el modelo ya entrenado, se clasifica un lote que el modelo no vio y se compara el conteo por clase con el real.

In [ ]:
lote = i_val[:150]
pred_lote = modelo.predict(X[lote])
inv = pd.DataFrame({"clase": range(10),
                    "real en el lote": [int((y[lote]==c_).sum()) for c_ in range(10)],
                    "el modelo dice": [int((pred_lote==c_).sum()) for c_ in range(10)]})
display(inv)
print("Coinciden en la imagen:", int((pred_lote==y[lote]).sum()), "de", len(lote))

## Parte 6 · Tu turno: tu ficha convertida en prompt
Ficha (ejemplo): *Camino B. Cambio: más imágenes de entrenamiento. Medida: k de N en validación. Criterio: me quedo con el cambio si gana por 5 o más. Riesgo: pocas imágenes por clase.*

Prompt para tu IA (cambia lo que está entre corchetes):
```
Tengo [N] imágenes de [qué] en carpetas por clase ([lista de clases]).
Hazme un notebook de Colab que: (1) las cargue y las reduzca a [tamaño],
(2) permita buscar las 5 más parecidas a una imagen, (3) entrene un clasificador
y diga cuántas acertó de cuántas en imágenes que no vio, (4) muestre las que falló.
Explica cada celda sin matemáticas. No cambies nada más de lo que te pedí.
```

**Con fotos reales** (productos, equipos, documentos) las imágenes de 8×8 no bastan: se usa un modelo de visión ya entrenado que convierte cada foto en una lista de números (como hicimos con texto en la clase 5 con el *encoder*) y sobre esa lista se hace lo mismo de las Partes 2 y 3. Esa variante **no está probada en este notebook**: pídele a tu IA que la agregue y mide con «k de N» antes de creerle.

Cuando termines: anota tu resultado en «k de N», 5 casos que probaste tú y el único cambio que harías después.